# 🎨 AI Identity Transformation Studio (Google Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

Welcome to the **AI Identity Transformation Studio** GPU inference hub.
This notebook provides state-of-the-art deep learning and generative models for:
1. **Photorealistic Neural Face Swapping** (InsightFace InSwapper + Face Restoration).
2. **Generative AI Character Synthesis** (Stable Diffusion Inpainting + Facial Presets: Cyborg, Alien, Marble Statue, Anime).
3. **Interactive Gradio Studio UI** with side-by-side previews and direct downloads.

### ⚡ Step 1: Verify GPU Acceleration
Ensure your Colab runtime is set to GPU: **Runtime -> Change runtime type -> T4 GPU (or A100/V100)**.

In [ ]:
!nvidia-smi

### 📦 Step 2: Install Dependencies & Download Model Weights

In [ ]:
# Install high-performance libraries
!pip install -q diffusers transformers accelerate insightface onnxruntime-gpu mediapipe opencv-python gradio Pillow

# Download InSwapper ONNX model and Face Restoration weights
!mkdir -p models
!curl -L -s -o models/inswapper_128.onnx "https://huggingface.co/ezioroot/inswapper/resolve/main/inswapper_128.onnx"
!curl -L -s -o models/GFPGANv1.4.pth "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth"

print("Dependencies and model weights installed successfully.")

### 🧠 Step 3: Initialize Deep Learning Models

In [ ]:
import torch
import cv2
import numpy as np
from PIL import Image
import insightface
from insightface.app import FaceAnalysis
from diffusers import StableDiffusionInpaintPipeline
import mediapipe as mp

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Compute device: {device}")

# 1. Initialize InsightFace detector and recognizer
print("Loading InsightFace Analysis (ArcFace)...")
face_app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
face_app.prepare(ctx_id=0 if device == "cuda" else -1, det_size=(640, 640))

# 2. Initialize InSwapper ONNX
print("Loading InSwapper-128...")
swapper = insightface.model_zoo.get_model("models/inswapper_128.onnx", download=False, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])

# 3. Initialize Stable Diffusion Inpainting in FP16
print("Loading Stable Diffusion Inpainting...")
sd_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)
sd_pipe = sd_pipe.to(device)
if device == "cuda":
    sd_pipe.enable_attention_slicing()

print("\nAll AI models loaded and ready for inference!")

### 🛠️ Step 4: Core Transformation Pipelines

In [ ]:
STYLE_PRESETS = {
    "Cybernetic Cyborg": "high-tech futuristic cybernetic cyborg android face, glowing blue circuitry under chrome skin, metallic plating, 8k, cinematic",
    "Extraterrestrial Alien": "celestial alien humanoid with iridescent bioluminescent skin, deep cosmic eyes, delicate extraterrestrial features, masterpiece",
    "Marble Statue": "classical Greco-Roman marble sculpture portrait, carved white Carrara marble with fine mineral veins, museum lighting, antique masterpiece",
    "Watercolor Anime": "delicate watercolor and ink illustration of a fantasy character face, soft pastel color wash, clean anime line art, Studio Ghibli style",
    "Steampunk Mechanoid": "Victorian steampunk automaton face, brass gears and clockwork mechanisms embedded in porcelain skin, ornate filigree, 8k",
}

def extract_facial_mask(image_bgr):
    h, w = image_bgr.shape[:2]
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    mp_mesh = mp.solutions.face_mesh
    with mp_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, min_detection_confidence=0.3) as mesh:
        res = mesh.process(rgb)
        if not res.multi_face_landmarks:
            raise ValueError("No face detected in target image.")
        pts = []
        for lm in res.multi_face_landmarks[0].landmark:
            pts.append([int(min(lm.x * w, w - 1)), int(min(lm.y * h, h - 1))])
        pts = np.array(pts, dtype=np.int32)
        
    hull = cv2.convexHull(pts)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillConvexPoly(mask, hull, 255)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
    mask = cv2.dilate(mask, kernel)
    mask = cv2.GaussianBlur(mask, (21, 21), 0)
    return mask

def run_neural_face_swap(source_img, target_img):
    """Photorealistic face swap via InsightFace InSwapper."""
    src_bgr = cv2.cvtColor(np.array(source_img), cv2.COLOR_RGB2BGR)
    tgt_bgr = cv2.cvtColor(np.array(target_img), cv2.COLOR_RGB2BGR)
    
    src_faces = face_app.get(src_bgr)
    tgt_faces = face_app.get(tgt_bgr)
    
    if not src_faces:
        raise ValueError("No face detected in Source Image.")
    if not tgt_faces:
        raise ValueError("No face detected in Target Image.")
        
    src_face = src_faces[0]
    res_bgr = tgt_bgr.copy()
    
    for tgt_face in tgt_faces:
        res_bgr = swapper.get(res_bgr, tgt_face, src_face, paste_back=True)
        
    return Image.fromarray(cv2.cvtColor(res_bgr, cv2.COLOR_BGR2RGB))

def run_generative_transformation(target_img, style_choice, custom_prompt, denoise_strength=0.8):
    """Generative character transformation via Stable Diffusion Inpainting."""
    tgt_bgr = cv2.cvtColor(np.array(target_img), cv2.COLOR_RGB2BGR)
    h, w = tgt_bgr.shape[:2]
    
    mask = extract_facial_mask(tgt_bgr)
    pil_mask = Image.fromarray(mask)
    pil_target = Image.fromarray(cv2.cvtColor(tgt_bgr, cv2.COLOR_BGR2RGB))
    
    prompt = custom_prompt if custom_prompt.strip() else STYLE_PRESETS.get(style_choice, STYLE_PRESETS["Cybernetic Cyborg"])
    neg_prompt = "monstrous, grotesque, blurry, messy, human, lowres, deformed, bad anatomy, extra limbs"
    
    out_pil = sd_pipe(
        prompt=prompt,
        negative_prompt=neg_prompt,
        image=pil_target,
        mask_image=pil_mask,
        guidance_scale=8.0,
        strength=denoise_strength,
    ).images[0]
    
    # Exact background preservation
    raw_out = cv2.cvtColor(np.array(out_pil.resize((w, h))), cv2.COLOR_RGB2BGR)
    soft_m = cv2.GaussianBlur(mask, (15, 15), 0).astype(np.float32) / 255.0
    soft_m_3ch = cv2.merge([soft_m, soft_m, soft_m])
    final_bgr = raw_out.astype(np.float32) * soft_m_3ch + tgt_bgr.astype(np.float32) * (1.0 - soft_m_3ch)
    final_bgr = np.clip(final_bgr, 0, 255).astype(np.uint8)
    
    return Image.fromarray(cv2.cvtColor(final_bgr, cv2.COLOR_BGR2RGB))

### 🚀 Step 5: Launch Interactive Gradio Studio UI
Run this cell to start the interactive web app. Click the **public URL** to share or open in a browser tab!

In [ ]:
import gradio as gr

with gr.Blocks(title="AI Identity Transformation Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎭 AI Identity Transformation Studio")
    gr.Markdown("**Transform faces with photorealistic deep learning swapping or generative AI synthesis.**")
    
    with gr.Tab("⚡ Photorealistic Neural Face Swap (InSwapper)"):
        with gr.Row():
            with gr.Column():
                swap_src = gr.Image(label="Source Face (Provides Identity)", type="pil")
                swap_tgt = gr.Image(label="Target Image (Provides Scene/Pose)", type="pil")
                swap_btn = gr.Button("Execute Photorealistic Swap", variant="primary")
            with gr.Column():
                swap_out = gr.Image(label="Swapped Result", type="pil")
        swap_btn.click(fn=run_neural_face_swap, inputs=[swap_src, swap_tgt], outputs=swap_out)
        
    with gr.Tab("🤖 Generative AI Character Transformation"):
        with gr.Row():
            with gr.Column():
                gen_tgt = gr.Image(label="Target Portrait Image", type="pil")
                style_dropdown = gr.Dropdown(choices=list(STYLE_PRESETS.keys()), value="Cybernetic Cyborg", label="Character Style Preset")
                custom_prompt_box = gr.Textbox(label="Custom Prompt (Optional)", placeholder="Leave blank to use selected preset")
                denoise_slider = gr.Slider(minimum=0.5, maximum=0.95, value=0.80, step=0.05, label="Transformation Strength")
                gen_btn = gr.Button("Generate Character Transformation", variant="primary")
            with gr.Column():
                gen_out = gr.Image(label="Transformed Result", type="pil")
        gen_btn.click(fn=run_generative_transformation, inputs=[gen_tgt, style_dropdown, custom_prompt_box, denoise_slider], outputs=gen_out)

# Launch with public sharing link enabled
demo.launch(share=True, debug=False)